# 03 — HMM Analysis (P3 Leka)

This notebook walks through:
1. Loading candidate data for a sample trip
2. Running the Viterbi decoder (distance-only emissions)
3. Visualising the decoded route on a Folium map
4. Comparing HMM vs classifier-only metrics
5. Parameter sensitivity analysis


In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathsnap.hmm.viterbi import decode
from pathsnap.evaluation.route_metrics import route_continuity, sequence_metrics, candidate_miss_rate
from pathsnap.viz.maps import make_trip_map, build_arc_segments
import json
from pathlib import Path

ROOT = Path('..').resolve()
print('Root:', ROOT)

In [ ]:
# --- Select a trip ---
TRIP_ID = '00000052'
SPLIT = 'student_test'

trip_dir = ROOT / 'data' / 'raw' / SPLIT / TRIP_ID

# Load raw files
nodes = pd.read_csv(trip_dir / f'{TRIP_ID}.nodes', header=None, names=['lon','lat'], sep=r'\s+')
arcs  = pd.read_csv(trip_dir / f'{TRIP_ID}.arcs',  header=None, names=['from_node','to_node'], sep=r'\s+')
track = pd.read_csv(trip_dir / f'{TRIP_ID}.track', header=None, names=['lon','lat','t'], sep=r'\s+')
print(f'Track: {len(track)} obs | Nodes: {len(nodes)} | Arcs: {len(arcs)}')

In [ ]:
# --- Load candidate parquet (if built) ---
cands_path = ROOT / 'data' / 'processed' / f'{SPLIT}_candidates.parquet'
if cands_path.exists():
    cands = pd.read_parquet(cands_path)
    cands = cands[cands['trip_id'] == TRIP_ID].copy()
    print(f'Candidates loaded: {len(cands)} rows')
else:
    print('No processed parquet found — run scripts/build_dataset.py first')
    cands = pd.DataFrame()

In [ ]:
# --- Build a minimal network object ---
import math

lat0 = nodes['lat'].mean()
m_per_deg_lat = 111_132.0
m_per_deg_lon = 111_132.0 * abs(math.cos(math.radians(lat0)))

fn = arcs['from_node'].to_numpy()
tn = arcs['to_node'].to_numpy()
fn_lat = nodes.iloc[fn]['lat'].to_numpy()
fn_lon = nodes.iloc[fn]['lon'].to_numpy()
tn_lat = nodes.iloc[tn]['lat'].to_numpy()
tn_lon = nodes.iloc[tn]['lon'].to_numpy()
arcs['length_m'] = np.sqrt(
    ((tn_lat - fn_lat) * m_per_deg_lat)**2 + ((tn_lon - fn_lon) * m_per_deg_lon)**2
).astype(np.float32)

out_arcs = {}
for arc_id, row in arcs.iterrows():
    out_arcs.setdefault(int(row['from_node']), []).append(int(arc_id))

class Net:
    def __init__(self):
        self.arcs_df = arcs
        self.nodes_df = nodes
        self.out_arcs = out_arcs

network = Net()
print('Network built')

In [ ]:
# --- Decode (v1: distance-only) ---
if not cands.empty:
    params = {'sigma_m': 20.0, 'beta': 20.0, 'alpha': 0.0, 'gap_s': 60.0, 'cutoff_m': 2000.0, 'max_obs': 5000}
    import time
    t0 = time.perf_counter()
    pred = decode(cands, network, params)
    print(f'Decoded {len(pred)} observations in {time.perf_counter()-t0:.2f}s')
    print(pred.head())
else:
    pred = None

In [ ]:
# --- Metrics ---
if pred is not None:
    cont = route_continuity(pred['arc_id'].tolist(), network)
    print(f'Route continuity: {cont:.4f}')

    try:
        ref = np.loadtxt(str(trip_dir / f'{TRIP_ID}.route'), dtype=np.int32)
        seq = sequence_metrics(pred['arc_id'].tolist(), ref.tolist())
        print('Sequence metrics:', seq)
    except FileNotFoundError:
        print('No .route file available')

In [ ]:
# --- Folium map ---
gps_latlon = [[float(r.lat), float(r.lon)] for r in track.itertuples()]

# Near-arc context
net_segs = build_arc_segments(list(range(min(3000, len(arcs)))), nodes, arcs, max_segs=3000)

if pred is not None:
    pred_segs = build_arc_segments(pred['arc_id'].tolist(), nodes, arcs)
else:
    pred_segs = None

try:
    ref = np.loadtxt(str(trip_dir / f'{TRIP_ID}.route'), dtype=np.int32)
    ref_segs = build_arc_segments(ref.tolist(), nodes, arcs)
except FileNotFoundError:
    ref_segs = None

m = make_trip_map(
    gps_latlon=gps_latlon,
    network_segments=net_segs,
    reference_segments=ref_segs,
    predicted_segments=pred_segs,
    title=f'PathSnap — Trip {TRIP_ID}',
)
m